## This notebook serves as a demonstration of the LASP SDTP application

#### Some commands to make things easier to test within this notebook

In [1]:
import glob
import json
from pathlib import Path

from lasp_sdtp.config import admin_config
from lasp_sdtp.database.database_controller import db
from lasp_sdtp.database.ingest import Ingest
from tests import conftest

# Clear out contents of database
db.session.query(db.FileQueue).delete()
db.session.query(db.Transactions).delete()
db.session.query(db.TagsAndExtras).delete()
db.session.query(db.Files).delete()
db.session.query(db.MissionShortnameMapping).delete()
db.session.query(db.Shortnames).delete()
db.session.query(db.MissionAccountMapping).delete()
db.session.query(db.Missions).delete()
db.session.commit()

# Add entries to database tables to support tests
conftest._add_missions_entries()
conftest._add_shortnames_entries()
conftest._add_mission_shortname_mapping_entries()
conftest._add_files_entries()

# Ingest testing data
filelist = glob.glob(str(Path(admin_config['filesystem_loc']) / 'prod' / '*'))
production = Ingest(filelist, 'prod', '01')
production.ingest()
development = Ingest(filelist, 'dev', '01')
development.ingest()

## Configuration

#### Subscribers must configure several parameters prior to using the `lasp_sdtp` application.  To configure a subscriber, instantiate the SubsciberConfig class, set the necessary parameters, and save.

In [2]:
from lasp_sdtp.config import SubscriberConfig

In [3]:
config = SubscriberConfig()

config.account_expiration_period = 1800  # The number of days the subscriber account will be valid
config.checksum_type = 'sha256'  # The type of checksum that the subscriber will use to validate responses
config.distinguished_name = ''  # The distingushed name to use in authentication
config.expiration_period = 180  # The number of days files will remain in the file queue
config.max_num_files = 10000  # The maximum number of files to return in file list requests
config.missions = ['TSIS2']
config.num_download_threads = 5  # The maximum number of simultaneous downloads
config.username = 'ges_disc'  # The subscriber username (used in authentication)

# Define the file stream(s) used for file transfer, along with any unique extras and tags
config.streams = {
    "prod": {
        "extras": {"aperture": {"type": "str", "default": ""}},
        "tags": {"observation_date": {"type": "str", "default": ""}}
    },
    "dev": {
        "extras": {"was_extrapolated": {"type": "bool", "default": ""}},
        "tags": {"irradiance": {"type": "float", "default": ""}}
    }
}

In [4]:
config.__dict__

{'account_expiration_period': 1800,
 'checksum_type': 'sha256',
 'distinguished_name': '',
 'expiration_period': 180,
 'max_num_files': 10000,
 'missions': ['TSIS2'],
 'num_download_threads': 5,
 'username': 'ges_disc',
 'streams': {'prod': {'extras': {'aperture': {'type': 'str', 'default': ''}},
   'tags': {'observation_date': {'type': 'str', 'default': ''}}},
  'dev': {'extras': {'was_extrapolated': {'type': 'bool', 'default': ''}},
   'tags': {'irradiance': {'type': 'float', 'default': ''}}}}}

In [5]:
config.save()

Checking to see if config is valid
Configuration is valid.
data/subscriber_config.json already exists. Are you sure you want to overwrite? (y/n)
y

Configuration file saved to data/subscriber_config.json


## Register Account

Register a new account.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned.
    
Command: `curl PUT "http://127.0.0.1:8000/register -H "Cert-UID: <certificate>"`

In [6]:
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## List Available Files

Return a list of available files given user-supplied parameters.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with a list of the available files and their metadata.  If no parameters are supplied, then all available files are returned.

Command: `curl GET "http://127.0.0.1:8000/files?param1=value1&param2=value2... -H "Cert-UID: <certificate>"`

#### Supported default parameters:
- `stream` (e.g. `prod`)
- `ShortName` (e.g. `TSIS_SC_L2`)
- `version` (e.g. `01`)

Subscriber-provided tags that are configured in the `subscriber_config` (e.g. `observation_date`) may also be used.

#### Examples

In [7]:
# List all available files
! curl -i -X GET "http://127.0.0.1:8000/files" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "available": true, 
      "checksum": "sha256:7uvyn7iak1rwob5g149y2t8ptrb68r7n3xx90jpt53tp15e2hmp8axqvz7qytvxv", 
      "deletion_date": null, 
      "expires": "2023-06-11", 
      "fileid": 1, 
      "ingest_date": "2021-12-31", 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "size": 55.0, 
      "tags": {
        "shortname": "TSIS_SC_L2", 
        "stream": "prod", 
        "version": "01"
      }
    }, 
    {
      "available": true, 
      "checksum": "sha256:0jc64zfsrsbm15jw1aeejouchcajz03nfsur6j083esy9l06ra85ptaeh49ia68j", 
      "deletion_date": null, 
      "expires": "2023-06-11", 
      "fileid": 2, 
      "ingest_date": "2022-01-01", 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "size": 55.0, 
      "tags": {
        "shortname": "TSIS_SC_L2", 
        "stream": "prod", 
        "version": "01"
      }
    }, 
    {
      "available": true, 
      "checksum": "sha256:0cfp4k2jplw1q25lmtk8oblqnblxksd3zo9cfpv5

In [8]:
# List all available files with ShortName of 'TSIS_SC_L2'
! curl -i -X GET "http://127.0.0.1:8000/files?shortname=TSIS_SC_L2" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "available": true, 
      "checksum": "sha256:7uvyn7iak1rwob5g149y2t8ptrb68r7n3xx90jpt53tp15e2hmp8axqvz7qytvxv", 
      "deletion_date": null, 
      "expires": "2023-06-11", 
      "fileid": 1, 
      "ingest_date": "2021-12-31", 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "size": 55.0, 
      "tags": {
        "shortname": "TSIS_SC_L2", 
        "stream": "prod", 
        "version": "01"
      }
    }, 
    {
      "available": true, 
      "checksum": "sha256:0jc64zfsrsbm15jw1aeejouchcajz03nfsur6j083esy9l06ra85ptaeh49ia68j", 
      "deletion_date": null, 
      "expires": "2023-06-11", 
      "fileid": 2, 
      "ingest_date": "2022-01-01", 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "size": 55.0, 
      "tags": {
        "shortname": "TSIS_SC_L2", 
        "stream": "prod", 
        "version": "01"
      }
    }, 
    {
      "available": true, 
      "checksum": "sha256:0cfp4k2jplw1q25lmtk8oblqnblxksd3zo9cfpv5

In [9]:
# List all available files with specific ShortName and for a specific stream
! curl -i -X GET "http://127.0.0.1:8000/files?shortname=TSIS_SC_L2&stream=dev" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "available": true, 
      "checksum": "sha256:jwildigq8oktoiomn4685oofti41ylsrv4xr4fzo7m410gnx8m43db40br0s5v05", 
      "deletion_date": null, 
      "expires": "2023-06-11", 
      "extras": {
        "was_extrapolated": "some_value"
      }, 
      "fileid": 120, 
      "ingest_date": "2021-12-31", 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "size": 55.0, 
      "tags": {
        "irradiance": "some_value", 
        "shortname": "TSIS_SC_L2", 
        "stream": "dev", 
        "version": "01"
      }
    }, 
    {
      "available": true, 
      "checksum": "sha256:jnv63bqojuc2vaxnd3lg0pu3d6jlc6pm7wz22zsc9vbouz5d8tzonmj0g24m54xg", 
      "deletion_date": null, 
      "expires": "2023-06-11", 
      "extras": {
        "was_extrapolated": "some_value"
      }, 
      "fileid": 121, 
      "ingest_date": "2022-01-01", 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "size": 55.0, 
      "tags": {
        "irradiance": "

In [10]:
# List all available files filtered by subscriber-specific tag
! curl -i -X GET "http://127.0.0.1:8000/files?shortname=TSIS_SC_L2&stream=dev&irradiance=some_value" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 2754
SDTP-TransactionID: 5
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Tue, 13 Dec 2022 18:42:55 GMT

{
  "files": [
    {
      "available": true, 
      "checksum": "sha256:jwildigq8oktoiomn4685oofti41ylsrv4xr4fzo7m410gnx8m43db40br0s5v05", 
      "deletion_date": null, 
      "expires": "2023-06-11", 
      "extras": {
        "was_extrapolated": "some_value"
      }, 
      "fileid": 120, 
      "ingest_date": "2021-12-31", 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "size": 55.0, 
      "tags": {
        "irradiance": "some_value", 
        "shortname": "TSIS_SC_L2", 
        "stream": "dev", 
        "version": "01"
      }
    }, 
    {
      "available": true, 
      "checksum": "sha256:jnv63bqojuc2vaxnd3lg0pu3d6jlc6pm7wz22zsc9vbouz5d8tzonmj0g24m54xg", 
      "deletion_date": null, 
      "expires": "2023-06-11", 
      "extras": {
        "was_extrapolated": "some_value"
      }, 
      "fileid": 1

## Get Specific File

Return the contents of a file for a given `fileid`.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with the content of the file.  

Command: `curl GET "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [12]:
! curl -i -X GET "http://127.0.0.1:8000/files/11" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220105_20220106.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220105_20220106.zip"
}


## Delete Specific File

Remove a specific file from the queue.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned with the content of the file.  

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [13]:
! curl -i -X DELETE "http://127.0.0.1:8000/files/11" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Delete Range of Files

Remove a range of files from the queue.  The user must supply the `Cert-UID` in the header of the request.  The range of files should be positive integers that increase in value.  If successful, a `204` response is returned.

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid_start>-<fileid_end> -H "Cert-UID: <certificate>"`

In [14]:
# First get a few files so that they are in the queue
! curl -i -X GET "http://127.0.0.1:8000/files/12" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/13" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/14" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 152
SDTP-TransactionID: 9
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Tue, 13 Dec 2022 18:43:44 GMT

{
  "contents": [
    "File contents for tsis2_ssi_L3_c12h_v01_20220102_20220103.nc"
  ], 
  "filename": "tsis2_ssi_L3_c12h_v01_20220102_20220103.nc"
}
HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 126
SDTP-TransactionID: 10
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Tue, 13 Dec 2022 18:43:44 GMT

{
  "contents": [
    "File contents for tsis2_sim_L2_v01_20220103.zip"
  ], 
  "filename": "tsis2_sim_L2_v01_20220103.zip"
}
HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 154
SDTP-TransactionID: 11
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Tue, 13 Dec 2022 18:43:44 GMT

{
  "contents": [
    "File contents for tsis2_tsi_L3_c06h_v01_20220105_20220106.txt"
  ], 
  "filename": "tsis2_tsi_L3_c06h_v01_20220105_20220106.txt"
}


In [15]:
# Delete the range of files
! curl -i -X DELETE "http://127.0.0.1:8000/files/12-14" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Examples of bad/invalid requests

Below are some examples of requests that will return an error

In [16]:
# Send request to endpoint that doesn't exist
# Expect a 404 response
! curl -i -X GET "http://127.0.0.1:8000/bogus_endpoint" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [17]:
# Try to register a closed account
# Expect a 401 response
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

401


In [18]:
# Request a file list with bogus parameters
# Expect a 400 response
! curl -i -X GET "http://127.0.0.1:8000/files?bogus_param=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files?shortname=TSIS_SC_L2&bogus_param=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files?date=20210101" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400


In [19]:
# Request a file list with valid parameters but no files returned
# Expect a 200 response with empty file list
! curl -i -X GET "http://127.0.0.1:8000/files?shortname=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": []
}


In [20]:
# Request a file that doesn't exist
# Expect a 404 response
! curl -i -X GET "http://127.0.0.1:8000/files/1234567" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [21]:
# Request a file with invalid fileid
# Expect a 400 response
! curl -i -X GET "http://127.0.0.1:8000/files/-1" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/1.5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/9999999999999999" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400
400


In [22]:
# Try to delete a file that doesn't exist
# Expect a 404 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/1234567" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [23]:
# Try to delete a file with invalid fileid
# Expect a 400 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/-1" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/1.5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400


In [24]:
# Try to delete a range of files that don't exist
# Expect a 404 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/1234567-1234569" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [25]:
# Try to delete a range of files with invalid fileids
# Expect a 400 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/foo-bar" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/10-5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/123-foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/10-10" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400
400


In [26]:
# Try to request a file that user doesn't have access to (file 23456 is hard-coded to be restricted)
# Expect a 403 response
! curl -i -X GET "http://127.0.0.1:8000/files/23456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

403


In [27]:
# Try to delete a file that user doesn't have access to (file 23456 is hard-coded to be restricted)
# Expect a 403 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/23456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

403


In [ ]:
# Send A LOT of requests at once